In [1]:

from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np
from IPython.display import display

# Define the directory containing the original datasets
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

USERS_PATH = DATA_DIR / "users.xml"

# Verify that the file exists
if not USERS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {USERS_PATH}"
    )

print("Dataset path:", USERS_PATH)
print(
    "File size (MB):",
    round(USERS_PATH.stat().st_size / (1024 ** 2), 2)
)

# Parse the XML incrementally to reduce memory usage
records = []

context = ET.iterparse(
    USERS_PATH,
    events=("start", "end")
)

_, root = next(context)

for event, elem in context:
    if event == "end" and elem.tag == "row":

        record = {
            child.tag: child.text
            for child in elem
        }

        records.append(record)

        # Release processed XML elements from memory
        root.clear()

# Convert extracted records into a DataFrame
df_users = pd.DataFrame.from_records(records)

# Preserve the original dataset for comparison
df_users_raw = df_users.copy()

print("\nDataset shape:", df_users.shape)

print("\nColumn names:")
print(df_users.columns.tolist())

display(df_users.head())


Dataset path: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\users.xml
File size (MB): 216.98

Dataset shape: (487292, 9)

Column names:
['id_user', 'timestamp_user_creation', 'username', 'follower_count', 'following_count', 'profile_url', 'finished_follower_scrape', 'finished_following_scrape', 'finished_truth_scrape']


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape
0,1238,2022-02-01 00:00:00,jeffgolfs,886,204,https://truthsocial.com/@jeffgolfs,f,f,f
1,50,2022-02-01 00:00:00,MizW,2365,2929,https://truthsocial.com/@MizW,f,f,f
2,6951,2022-02-01 00:00:00,SusanCreech,870,1232,https://truthsocial.com/@SusanCreech,f,f,f
3,9861,2022-02-01 00:00:00,BMF6FeeTDeeP,1086,1654,https://truthsocial.com/@BMF6FeeTDeeP,f,f,f
4,34,2022-02-01 00:00:00,DC_Draino,286146,81,https://truthsocial.com/@DC_Draino,t,t,t


In [2]:

# Define a summary of initial data quality
quality_baseline = pd.DataFrame({
    "dtype": df_users.dtypes.astype(str),
    "null_count": df_users.isna().sum(),
    "null_percentage": (
        df_users.isna().mean() * 100
    ).round(2),
    "sentinel_count": [
        df_users[col].astype(str).str.strip().eq("-1").sum()
        for col in df_users.columns
    ],
    "unique_values": df_users.nunique(dropna=True)
})

# Calculate the percentage of sentinel values
quality_baseline["sentinel_percentage"] = (
    quality_baseline["sentinel_count"]
    / len(df_users) * 100
).round(2)

display(quality_baseline)

print("\nTotal records:", len(df_users))
print("Duplicated user IDs:",
      df_users["id_user"].duplicated().sum())
print("Completely duplicated records:",
      df_users.duplicated().sum())


,dtype,null_count,null_percentage,sentinel_count,unique_values,sentinel_percentage
id_user,str,0,0.0,0,487292,0.00
timestamp_user_creation,str,0,0.0,465337,16,95.49
username,str,0,0.0,18,487153,0.00
follower_count,str,0,0.0,465337,8695,95.49
following_count,str,0,0.0,465337,7313,95.49
profile_url,str,0,0.0,0,487150,0.00
finished_follower_scrape,str,0,0.0,465337,3,95.49
finished_following_scrape,str,0,0.0,465337,3,95.49
finished_truth_scrape,str,0,0.0,465337,3,95.49



Total records: 487292
Duplicated user IDs: 0
Completely duplicated records: 0


In [3]:

# Identify columns containing sentinel values
sentinel_columns = [
    "timestamp_user_creation",
    "follower_count",
    "following_count",
    "finished_follower_scrape",
    "finished_following_scrape",
    "finished_truth_scrape"
]

# Build a boolean matrix identifying sentinel values
sentinel_mask = df_users[sentinel_columns].eq("-1")

# Count sentinel values per user
df_users["sentinel_count"] = sentinel_mask.sum(axis=1)

print("Distribution of sentinel values per user:")
display(
    df_users["sentinel_count"]
    .value_counts()
    .sort_index()
    .rename_axis("sentinel_count")
    .reset_index(name="number_of_users")
)

# Analyze combinations of missing attributes
missing_patterns = (
    sentinel_mask
    .value_counts()
    .reset_index(name="number_of_users")
    .sort_values("number_of_users", ascending=False)
)

missing_patterns["percentage"] = (
    missing_patterns["number_of_users"]
    / len(df_users) * 100
).round(2)

print("\nMost frequent missing-value patterns:")
display(missing_patterns.head(10))

# Verify whether sentinel values always occur together
all_missing = sentinel_mask.all(axis=1)
none_missing = ~sentinel_mask.any(axis=1)
partial_missing = ~(all_missing | none_missing)

print("\nUsers with all six sentinel values:", all_missing.sum())
print("Users with no sentinel values:", none_missing.sum())
print("Users with partial sentinel values:", partial_missing.sum())


Distribution of sentinel values per user:


,sentinel_count,number_of_users
0,0,21955
1,6,465337



Most frequent missing-value patterns:


,timestamp_user_creation,follower_count,following_count,finished_follower_scrape,finished_following_scrape,finished_truth_scrape,number_of_users,percentage
0,True,True,True,True,True,True,465337,95.49
1,False,False,False,False,False,False,21955,4.51



Users with all six sentinel values: 465337
Users with no sentinel values: 21955
Users with partial sentinel values: 0


In [4]:

# Classify records based on sentinel values
df_users["data_completeness"] = np.select(
    [
        df_users["sentinel_count"].eq(0),
        df_users["sentinel_count"].eq(6)
    ],
    [
        "Complete",
        "Incomplete"
    ],
    default="Partially incomplete"
)

# Summarize the groups
completeness_summary = (
    df_users.groupby("data_completeness")
    .agg(
        total_users=("id_user", "size"),
        unique_user_ids=("id_user", "nunique"),
        unique_usernames=("username", "nunique"),
        unique_profile_urls=("profile_url", "nunique")
    )
)

completeness_summary["percentage"] = (
    completeness_summary["total_users"]
    / len(df_users) * 100
).round(2)

display(completeness_summary)

# Display examples from incomplete records
print("\nExamples of incomplete users:")
display(
    df_users.loc[
        df_users["data_completeness"] == "Incomplete"
    ].head(10)
)


,total_users,unique_user_ids,unique_usernames,unique_profile_urls,percentage
data_completeness,,,,,
Complete,21955,21955,21955,21955,4.51
Incomplete,465337,465337,465198,465198,95.49



Examples of incomplete users:


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape,sentinel_count,data_completeness
21955,22025,-1,hazel_eyez,-1,-1,https://truthsocial.com/@hazel_eyez,-1,-1,-1,6,Incomplete
21956,22026,-1,Trucker55,-1,-1,https://truthsocial.com/@Trucker55,-1,-1,-1,6,Incomplete
21957,22027,-1,foxxed,-1,-1,https://truthsocial.com/@foxxed,-1,-1,-1,6,Incomplete
21958,22028,-1,RyderMcCue,-1,-1,https://truthsocial.com/@RyderMcCue,-1,-1,-1,6,Incomplete
21959,22029,-1,TruePatriotNurse21,-1,-1,https://truthsocial.com/@TruePatriotNurse21,-1,-1,-1,6,Incomplete
21960,22030,-1,myteam,-1,-1,https://truthsocial.com/@myteam,-1,-1,-1,6,Incomplete
21961,22031,-1,JB2ndID,-1,-1,https://truthsocial.com/@JB2ndID,-1,-1,-1,6,Incomplete
21962,22032,-1,ConstitutionalGranny,-1,-1,https://truthsocial.com/@ConstitutionalGranny,-1,-1,-1,6,Incomplete
21963,22033,-1,ArtLady17,-1,-1,https://truthsocial.com/@ArtLady17,-1,-1,-1,6,Incomplete
21964,22034,-1,Debfea,-1,-1,https://truthsocial.com/@Debfea,-1,-1,-1,6,Incomplete


In [5]:

# Convert user IDs to numeric for analysis
df_users["id_user_numeric"] = pd.to_numeric(
    df_users["id_user"],
    errors="coerce"
)

# Compare user ID distributions across completeness groups
id_summary = (
    df_users.groupby("data_completeness")
    ["id_user_numeric"]
    .agg(["count", "min", "median", "max"])
)

print("User ID distribution by completeness group:")
display(id_summary)

# Check completeness distribution before and after the
# boundary observed in the dataset
boundary = 22025

df_users["id_range"] = np.where(
    df_users["id_user_numeric"] < boundary,
    "ID < 22025",
    "ID >= 22025"
)

id_range_summary = pd.crosstab(
    df_users["id_range"],
    df_users["data_completeness"]
)

print("\nCompleteness by ID range:")
display(id_range_summary)

# Check whether complete and incomplete records
# appear in separate blocks in the original XML
completeness_transitions = (
    df_users["data_completeness"]
    .ne(df_users["data_completeness"].shift())
)

print(
    "\nNumber of completeness blocks:",
    completeness_transitions.sum()
)

print("\nStarting positions of completeness blocks:")
display(
    df_users.loc[
        completeness_transitions,
        ["id_user", "data_completeness"]
    ].head(20)
)


User ID distribution by completeness group:


,count,min,median,max
data_completeness,,,,
Complete,21955,22,11047.0,22024
Incomplete,465337,22025,254693.0,487361



Completeness by ID range:


data_completeness,Complete,Incomplete
id_range,,
ID < 22025,21955,0
ID >= 22025,0,465337



Number of completeness blocks: 2

Starting positions of completeness blocks:


,id_user,data_completeness
0,1238,Complete
21955,22025,Incomplete


In [6]:

# Define the path to the follows dataset
FOLLOWS_PATH = DATA_DIR / "follows.json"

if not FOLLOWS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {FOLLOWS_PATH}"
    )

print("Follows dataset path:", FOLLOWS_PATH)

print(
    "File size (MB):",
    round(FOLLOWS_PATH.stat().st_size / (1024 ** 2), 2)
)

# Load follows dataset
df_follows = pd.read_json(FOLLOWS_PATH)

print("\nDataset shape:", df_follows.shape)

print("\nColumn names:")
print(df_follows.columns.tolist())

print("\nData types:")
display(df_follows.dtypes)

print("\nFirst five records:")
display(df_follows.head())


Follows dataset path: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\follows.json
File size (MB): 117.6



Dataset shape: (4002115, 2)

Column names:
['follower', 'followed']

Data types:


follower    int64
followed    int64
dtype: object


First five records:


,follower,followed
0,12475,22025
1,12475,22026
2,12475,22027
3,12475,22028
4,12475,17280


In [7]:

# Normalize user identifiers for cross-dataset comparison
user_ids = pd.to_numeric(
    df_users["id_user"],
    errors="coerce"
)

follower_ids = pd.to_numeric(
    df_follows["follower"],
    errors="coerce"
)

followed_ids = pd.to_numeric(
    df_follows["followed"],
    errors="coerce"
)

# Identify all unique users appearing in follows.json
network_users = pd.Index(
    pd.concat(
        [follower_ids, followed_ids],
        ignore_index=True
    ).dropna().unique()
)

# Check whether each user appears in the follows network
df_users["appears_in_follows"] = (
    user_ids.isin(network_users)
)

# Calculate coverage for each completeness group
coverage_summary = (
    df_users.groupby("data_completeness")
    ["appears_in_follows"]
    .agg(["sum", "count", "mean"])
)

coverage_summary.columns = [
    "users_in_follows",
    "total_users",
    "coverage_rate"
]

coverage_summary["coverage_percentage"] = (
    coverage_summary["coverage_rate"] * 100
).round(2)

display(coverage_summary)

print("\nTotal unique users in follows.json:",
      len(network_users))

print(
    "Users from users.xml appearing in follows.json:",
    df_users["appears_in_follows"].sum()
)

print(
    "Users from users.xml absent from follows.json:",
    (~df_users["appears_in_follows"]).sum()
)

print(
    "Network users absent from users.xml:",
    (~network_users.isin(user_ids.dropna())).sum()
)


,users_in_follows,total_users,coverage_rate,coverage_percentage
data_completeness,,,,
Complete,21801,21955,0.992986,99.30
Incomplete,423149,465337,0.909339,90.93



Total unique users in follows.json: 444951
Users from users.xml appearing in follows.json: 444950
Users from users.xml absent from follows.json: 42342
Network users absent from users.xml: 1


In [8]:

# Check the integrity of follow relationships
duplicate_edges = df_follows.duplicated(
    subset=["follower", "followed"]
).sum()

self_follows = (
    df_follows["follower"] == df_follows["followed"]
).sum()

print("Total relationships:", len(df_follows))
print("Duplicated relationships:", duplicate_edges)
print("Self-follow relationships:", self_follows)

# Create a separate working copy of valid relationships
df_follows_clean = (
    df_follows
    .dropna(subset=["follower", "followed"])
    .drop_duplicates(subset=["follower", "followed"])
    .copy()
)

# Exclude self-follow relationships from degree calculations
df_follows_clean = df_follows_clean.loc[
    df_follows_clean["follower"]
    != df_follows_clean["followed"]
]

# Count distinct observed followers for each user
observed_followers = (
    df_follows_clean
    .groupby("followed")["follower"]
    .nunique()
    .rename("observed_follower_count")
)

# Count distinct observed following accounts for each user
observed_following = (
    df_follows_clean
    .groupby("follower")["followed"]
    .nunique()
    .rename("observed_following_count")
)

# Create a comparison dataset without modifying original values
df_network_comparison = df_users[
    ["id_user", "data_completeness",
     "follower_count", "following_count"]
].copy()

df_network_comparison["id_user"] = pd.to_numeric(
    df_network_comparison["id_user"],
    errors="coerce"
)

df_network_comparison = df_network_comparison.merge(
    observed_followers,
    left_on="id_user",
    right_index=True,
    how="left"
)

df_network_comparison = df_network_comparison.merge(
    observed_following,
    left_on="id_user",
    right_index=True,
    how="left"
)

# Missing network degrees mean no relationships were observed
df_network_comparison[
    ["observed_follower_count", "observed_following_count"]
] = df_network_comparison[
    ["observed_follower_count", "observed_following_count"]
].fillna(0).astype("int64")

print("\nNetwork count comparison:")
display(df_network_comparison.head(10))

print("\nObserved network counts for incomplete users:")
display(
    df_network_comparison.loc[
        df_network_comparison["data_completeness"] == "Incomplete",
        ["observed_follower_count", "observed_following_count"]
    ].describe()
)


Total relationships: 4002115
Duplicated relationships: 99
Self-follow relationships: 0

Network count comparison:


,id_user,data_completeness,follower_count,following_count,observed_follower_count,observed_following_count
0,1238,Complete,886,204,17,0
1,50,Complete,2365,2929,81,1
2,6951,Complete,870,1232,17,0
3,9861,Complete,1086,1654,21,0
4,34,Complete,286146,81,981,87
5,40,Complete,1222,1402,63,1456
6,37,Complete,68063,21,262,22
7,22,Complete,455386,2,822,2
8,25,Complete,6942,221,53,0
9,52,Complete,1900,2131,71,1



Observed network counts for incomplete users:


,observed_follower_count,observed_following_count
count,465337.000000,465337.000000
mean,5.484232,0.005256
std,13.491696,2.602313
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,0.000000
75%,4.000000,0.000000
max,539.000000,1680.000000


In [9]:

# Select users with available original profile counts
complete_users = df_network_comparison.loc[
    df_network_comparison["data_completeness"] == "Complete"
].copy()

# Convert original counts to numeric
complete_users["follower_count"] = pd.to_numeric(
    complete_users["follower_count"],
    errors="coerce"
)

complete_users["following_count"] = pd.to_numeric(
    complete_users["following_count"],
    errors="coerce"
)

# Calculate differences between observed and original counts
complete_users["follower_difference"] = (
    complete_users["observed_follower_count"]
    - complete_users["follower_count"]
)

complete_users["following_difference"] = (
    complete_users["observed_following_count"]
    - complete_users["following_count"]
)

# Evaluate exact matches and inconsistencies
validation_summary = pd.DataFrame({
    "Metric": ["Followers", "Following"],
    "Exact matches": [
        complete_users["follower_difference"].eq(0).sum(),
        complete_users["following_difference"].eq(0).sum()
    ],
    "Observed lower": [
        complete_users["follower_difference"].lt(0).sum(),
        complete_users["following_difference"].lt(0).sum()
    ],
    "Observed higher": [
        complete_users["follower_difference"].gt(0).sum(),
        complete_users["following_difference"].gt(0).sum()
    ]
})

for col in ["Exact matches", "Observed lower", "Observed higher"]:
    validation_summary[col + " (%)"] = (
        validation_summary[col]
        / len(complete_users) * 100
    ).round(2)

display(validation_summary)

# Summarize discrepancies
print("\nFollower count differences:")
display(complete_users["follower_difference"].describe())

print("\nFollowing count differences:")
display(complete_users["following_difference"].describe())


,Metric,Exact matches,Observed lower,Observed higher,Exact matches (%),Observed lower (%),Observed higher (%)
0,Followers,99,21308,548,0.45,97.05,2.50
1,Following,2251,19179,525,10.25,87.36,2.39



Follower count differences:


count    2.195500e+04
mean    -7.245762e+03
std      5.626274e+04
min     -4.025987e+06
25%     -5.052000e+03
50%     -1.888000e+03
75%     -4.030000e+02
max      7.680000e+02
Name: follower_difference, dtype: float64


Following count differences:


count    21955.000000
mean     -3175.333318
std       5337.241052
min     -99514.000000
25%      -5051.500000
50%      -1270.000000
75%        -64.000000
max       2553.000000
Name: following_difference, dtype: float64

In [10]:
# Create a working copy for data cleaning
df_users_clean = df_users_raw.copy()

# Store initial dataset dimensions
initial_rows = len(df_users_clean)
initial_columns = len(df_users_clean.columns)

print("Initial dataset shape:", df_users_clean.shape)

print("\nOriginal data types:")
display(df_users_clean.dtypes.to_frame("dtype"))

print("\nFirst five records:")
display(df_users_clean.head())

Initial dataset shape: (487292, 9)

Original data types:


,dtype
id_user,str
timestamp_user_creation,str
username,str
follower_count,str
following_count,str
profile_url,str
finished_follower_scrape,str
finished_following_scrape,str
finished_truth_scrape,str



First five records:


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape
0,1238,2022-02-01 00:00:00,jeffgolfs,886,204,https://truthsocial.com/@jeffgolfs,f,f,f
1,50,2022-02-01 00:00:00,MizW,2365,2929,https://truthsocial.com/@MizW,f,f,f
2,6951,2022-02-01 00:00:00,SusanCreech,870,1232,https://truthsocial.com/@SusanCreech,f,f,f
3,9861,2022-02-01 00:00:00,BMF6FeeTDeeP,1086,1654,https://truthsocial.com/@BMF6FeeTDeeP,f,f,f
4,34,2022-02-01 00:00:00,DC_Draino,286146,81,https://truthsocial.com/@DC_Draino,t,t,t


In [11]:
# Define columns using -1 as a missing-value marker
missing_value_columns = [
    "timestamp_user_creation",
    "username",
    "follower_count",
    "following_count",
    "finished_follower_scrape",
    "finished_following_scrape",
    "finished_truth_scrape"
]

# Normalize whitespace and empty strings
for col in df_users_clean.columns:
    df_users_clean[col] = (
        df_users_clean[col]
        .astype("string")
        .str.strip()
    )

    df_users_clean[col] = (
        df_users_clean[col].replace("", pd.NA)
    )

# Replace sentinel values with proper missing values
for col in missing_value_columns:
    df_users_clean[col] = (
        df_users_clean[col].replace("-1", pd.NA)
    )

# Summarize missing values after normalization
missing_summary = pd.DataFrame({
    "missing_count": df_users_clean.isna().sum(),
    "missing_percentage": (
        df_users_clean.isna().mean() * 100
    ).round(2)
})

display(missing_summary)

print("\nTotal missing cells:",
      df_users_clean.isna().sum().sum())

,missing_count,missing_percentage
id_user,0,0.00
timestamp_user_creation,465337,95.49
username,18,0.00
follower_count,465337,95.49
following_count,465337,95.49
profile_url,0,0.00
finished_follower_scrape,465337,95.49
finished_following_scrape,465337,95.49
finished_truth_scrape,465337,95.49



Total missing cells: 2792040


In [12]:
# Convert user identifiers and counts to nullable integers
numeric_columns = [
    "id_user",
    "follower_count",
    "following_count"
]

for col in numeric_columns:
    df_users_clean[col] = (
        pd.to_numeric(
            df_users_clean[col],
            errors="coerce"
        ).astype("Int64")
    )

# Convert timestamps to datetime
df_users_clean["timestamp_user_creation"] = (
    pd.to_datetime(
        df_users_clean["timestamp_user_creation"],
        errors="coerce"
    )
)

# Convert scraping flags to nullable booleans
boolean_columns = [
    "finished_follower_scrape",
    "finished_following_scrape",
    "finished_truth_scrape"
]

boolean_mapping = {
    "t": True,
    "f": False
}

for col in boolean_columns:
    df_users_clean[col] = (
        df_users_clean[col]
        .map(boolean_mapping)
        .astype("boolean")
    )

# Display corrected data types
print("Data types after cleaning:")
display(
    df_users_clean.dtypes.to_frame("cleaned_dtype")
)

print("\nCleaned dataset preview:")
display(df_users_clean.head())

Data types after cleaning:


,cleaned_dtype
id_user,Int64
timestamp_user_creation,datetime64[us]
username,string
follower_count,Int64
following_count,Int64
profile_url,string
finished_follower_scrape,boolean
finished_following_scrape,boolean
finished_truth_scrape,boolean



Cleaned dataset preview:


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape
0,1238,2022-02-01,jeffgolfs,886,204,https://truthsocial.com/@jeffgolfs,False,False,False
1,50,2022-02-01,MizW,2365,2929,https://truthsocial.com/@MizW,False,False,False
2,6951,2022-02-01,SusanCreech,870,1232,https://truthsocial.com/@SusanCreech,False,False,False
3,9861,2022-02-01,BMF6FeeTDeeP,1086,1654,https://truthsocial.com/@BMF6FeeTDeeP,False,False,False
4,34,2022-02-01,DC_Draino,286146,81,https://truthsocial.com/@DC_Draino,True,True,True


In [13]:
# Check identifier integrity
missing_ids = df_users_clean["id_user"].isna().sum()

duplicate_ids = (
    df_users_clean["id_user"].duplicated().sum()
)

duplicate_records = (
    df_users_clean.duplicated().sum()
)

invalid_ids = (
    df_users_clean["id_user"]
    .dropna()
    .le(0)
    .sum()
)

# Summarize integrity checks
identifier_validation = pd.DataFrame({
    "Check": [
        "Missing user IDs",
        "Duplicated user IDs",
        "Completely duplicated records",
        "Non-positive user IDs"
    ],
    "Count": [
        missing_ids,
        duplicate_ids,
        duplicate_records,
        invalid_ids
    ]
})

display(identifier_validation)

# Verify whether the user ID is a valid primary key
valid_primary_key = (
    missing_ids == 0
    and duplicate_ids == 0
    and invalid_ids == 0
)

print("\nValid primary key:", valid_primary_key)

,Check,Count
0,Missing user IDs,0
1,Duplicated user IDs,0
2,Completely duplicated records,0
3,Non-positive user IDs,0



Valid primary key: True


In [14]:
# Count missing identifiers and profile information
missing_usernames = (
    df_users_clean["username"].isna().sum()
)

missing_urls = (
    df_users_clean["profile_url"].isna().sum()
)

# Count duplicated non-null usernames and URLs
duplicate_usernames = (
    df_users_clean["username"]
    .dropna()
    .duplicated()
    .sum()
)

duplicate_urls = (
    df_users_clean["profile_url"]
    .dropna()
    .duplicated()
    .sum()
)

# Extract account names from Truth Social profile URLs
url_username = (
    df_users_clean["profile_url"]
    .str.extract(
        r"(?i)^https?://(?:www\.)?truthsocial\.com/@([^/?#]+)/?$",
        expand=False
    )
)

# Compare usernames and URL account names
valid_comparison = (
    df_users_clean["username"].notna()
    & url_username.notna()
)

username_mismatch = (
    valid_comparison
    & (
        df_users_clean["username"].str.casefold()
        != url_username.str.casefold()
    )
)

# Identify non-null URLs not matching the expected format
invalid_url_format = (
    df_users_clean["profile_url"].notna()
    & url_username.isna()
)

# Summarize quality issues
profile_validation = pd.DataFrame({
    "Check": [
        "Missing usernames",
        "Missing profile URLs",
        "Duplicated usernames",
        "Duplicated profile URLs",
        "Username / URL mismatches",
        "Unexpected profile URL formats"
    ],
    "Count": [
        missing_usernames,
        missing_urls,
        duplicate_usernames,
        duplicate_urls,
        username_mismatch.sum(),
        invalid_url_format.sum()
    ]
})

display(profile_validation)

# Inspect inconsistent records
print("\nExamples of username / URL mismatches:")

display(
    df_users_clean.loc[
        username_mismatch,
        ["id_user", "username", "profile_url"]
    ].head(20)
)

print("\nExamples of unexpected URL formats:")

display(
    df_users_clean.loc[
        invalid_url_format,
        ["id_user", "username", "profile_url"]
    ].head(20)
)

,Check,Count
0,Missing usernames,18
1,Missing profile URLs,0
2,Duplicated usernames,122
3,Duplicated profile URLs,142
4,Username / URL mismatches,3
5,Unexpected profile URL formats,1



Examples of username / URL mismatches:


,id_user,username,profile_url
6239,7804,WSJPolitics,https://truthsocial.com/@cpac
7040,6384,bean8920,https://truthsocial.com/@dani
7250,6582,SiydenGalanter,https://truthsocial.com/@lovelynn



Examples of unexpected URL formats:


,id_user,username,profile_url
445410,445480,please take the time to like and/or ReTruth th...,https://truthsocial.com/@please take the time ...


In [15]:
# Define numerical attributes
count_columns = [
    "follower_count",
    "following_count"
]

outlier_results = []

for col in count_columns:

    values = df_users_clean[col].dropna()

    # Identify invalid negative counts
    negative_count = (values < 0).sum()

    # Calculate IQR thresholds
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)

    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Detect statistical outliers
    outliers = (
        (values < lower_bound)
        | (values > upper_bound)
    )

    outlier_results.append({
        "column": col,
        "valid_values": len(values),
        "negative_values": negative_count,
        "minimum": values.min(),
        "median": values.median(),
        "maximum": values.max(),
        "IQR_upper_bound": upper_bound,
        "IQR_outliers": outliers.sum(),
        "outlier_percentage": round(
            outliers.mean() * 100, 2
        )
    })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary)

print("\nNumerical distributions:")

display(
    df_users_clean[count_columns].describe(
        percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

print("\nTop 10 accounts by follower count:")

display(
    df_users_clean.nlargest(
        10, "follower_count"
    )[
        [
            "id_user",
            "username",
            "follower_count",
            "following_count"
        ]
    ]
)

,column,valid_values,negative_values,minimum,median,maximum,IQR_upper_bound,IQR_outliers,outlier_percentage
0,follower_count,21955,0,0,1929.0,4027829,12227.75,1656,7.54
1,following_count,21955,0,0,1540.0,99514,13247.50,617,2.81



Numerical distributions:


,follower_count,following_count
count,21955.0,21955.0
mean,7311.806286,3357.504578
std,56325.798118,5351.596336
min,0.0,0.0
25%,414.0,150.0
50%,1929.0,1540.0
75%,5139.5,5389.0
95%,17830.2,9381.6
99%,68801.32,25341.4
max,4027829.0,99514.0



Top 10 accounts by follower count:


,id_user,username,follower_count,following_count
19118,27,realDonaldTrump,4027829,3
564,300,DonaldJTrumpJr,2524434,227
246,160,truthsocial,2375491,2
85,105,dbongino,1955706,249
175,211,BabylonBee,1809193,1
78,26,EricTrump,1727967,178
284,145,NewsMax,1511293,48
288,322,DineshDSouza,1491112,301
863,103,seanhannity,1288592,6
129,147,rumble,1142497,58


In [16]:
# Prepare observed network counts from previous analysis
network_features = df_network_comparison[
    [
        "id_user",
        "observed_follower_count",
        "observed_following_count"
    ]
].copy()

# Ensure consistent identifier types
network_features["id_user"] = (
    network_features["id_user"].astype("Int64")
)

# Merge network features into the cleaned users dataset
df_users_clean = df_users_clean.merge(
    network_features,
    on="id_user",
    how="left",
    validate="one_to_one"
)

# Verify that the left join preserved all users
assert len(df_users_clean) == initial_rows

# Add information availability indicators
df_users_clean["has_profile_counts"] = (
    df_users_clean["follower_count"].notna()
    & df_users_clean["following_count"].notna()
)

df_users_clean["appears_in_follows"] = (
    df_users_clean["id_user"].isin(network_users)
)

# Explicitly mark observed counts as sample-derived features
df_users_clean["observed_follower_count"] = (
    df_users_clean["observed_follower_count"]
    .astype("Int64")
)

df_users_clean["observed_following_count"] = (
    df_users_clean["observed_following_count"]
    .astype("Int64")
)

print("Dataset shape after enrichment:",
      df_users_clean.shape)

print("\nEnriched user records:")

display(
    df_users_clean[
        [
            "id_user",
            "follower_count",
            "observed_follower_count",
            "following_count",
            "observed_following_count",
            "has_profile_counts",
            "appears_in_follows"
        ]
    ].head(10)
)

Dataset shape after enrichment: (487292, 13)

Enriched user records:


,id_user,follower_count,observed_follower_count,following_count,observed_following_count,has_profile_counts,appears_in_follows
0,1238,886,17,204,0,True,True
1,50,2365,81,2929,1,True,True
2,6951,870,17,1232,0,True,True
3,9861,1086,21,1654,0,True,True
4,34,286146,981,81,87,True,True
5,40,1222,63,1402,1456,True,True
6,37,68063,262,21,22,True,True
7,22,455386,822,2,2,True,True
8,25,6942,53,221,0,True,True
9,52,1900,71,2131,1,True,True


In [17]:
# Create the set of valid user identifiers
valid_user_ids = pd.Index(
    df_users_clean["id_user"].dropna().unique()
)

# Identify network users absent from users.xml
unmatched_network_users = network_users[
    ~network_users.isin(valid_user_ids)
]

# Identify users absent from the observed network
users_without_network_data = (
    ~df_users_clean["id_user"].isin(network_users)
).sum()

# Count relationships with missing user references
invalid_follower_refs = (
    ~df_follows_clean["follower"].isin(valid_user_ids)
)

invalid_followed_refs = (
    ~df_follows_clean["followed"].isin(valid_user_ids)
)

invalid_relationships = (
    invalid_follower_refs | invalid_followed_refs
)

# Summarize referential integrity
referential_integrity = pd.DataFrame({
    "Check": [
        "Network users absent from users.xml",
        "Users absent from follows.json",
        "Edges with unknown followers",
        "Edges with unknown followed users",
        "Edges with at least one unknown user"
    ],
    "Count": [
        len(unmatched_network_users),
        users_without_network_data,
        invalid_follower_refs.sum(),
        invalid_followed_refs.sum(),
        invalid_relationships.sum()
    ]
})

display(referential_integrity)

print("\nUnmatched network user IDs:")
print(unmatched_network_users.tolist())

print("\nExamples of unmatched relationships:")

display(
    df_follows_clean.loc[
        invalid_relationships
    ].head(10)
)

,Check,Count
0,Network users absent from users.xml,1
1,Users absent from follows.json,42342
2,Edges with unknown followers,0
3,Edges with unknown followed users,1
4,Edges with at least one unknown user,1



Unmatched network user IDs:
[97]

Examples of unmatched relationships:


,follower,followed
1159076,26,97


In [18]:
# Check timestamp validity
invalid_future_dates = (
    df_users_clean["timestamp_user_creation"]
    > pd.Timestamp.now()
).sum()

# Check negative profile counts
negative_followers = (
    df_users_clean["follower_count"]
    .dropna()
    .lt(0)
    .sum()
)

negative_following = (
    df_users_clean["following_count"]
    .dropna()
    .lt(0)
    .sum()
)

# Check identifier integrity
missing_ids = df_users_clean["id_user"].isna().sum()

duplicate_ids = (
    df_users_clean["id_user"].duplicated().sum()
)

# Build final validation report
final_validation = pd.DataFrame({
    "Check": [
        "Missing user IDs",
        "Duplicated user IDs",
        "Negative follower counts",
        "Negative following counts",
        "Future creation timestamps",
        "Duplicated records"
    ],
    "Count": [
        missing_ids,
        duplicate_ids,
        negative_followers,
        negative_following,
        invalid_future_dates,
        df_users_clean.duplicated().sum()
    ]
})

display(final_validation)

# Compare original and cleaned missing values
# Include original columns only for a fair comparison
original_columns = df_users_raw.columns.tolist()

raw_missing = pd.DataFrame({
    col: (
        df_users_raw[col].isna()
        | df_users_raw[col].astype("string")
          .str.strip().isin(["", "-1"])
    ).sum()
    for col in original_columns
}, index=["missing_count"]).T

clean_missing = (
    df_users_clean[original_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

quality_comparison = pd.DataFrame({
    "original_missing": raw_missing["missing_count"],
    "cleaned_missing": clean_missing["missing_count"]
})

quality_comparison["cleaned_missing_percentage"] = (
    quality_comparison["cleaned_missing"]
    / len(df_users_clean) * 100
).round(2)

print("\nMissing value comparison:")
display(quality_comparison)

print("\nFinal dataset shape:")
print(df_users_clean.shape)

# Assert essential structural invariants
assert len(df_users_clean) == initial_rows

assert df_users_clean["id_user"].notna().all()

assert df_users_clean["id_user"].is_unique

print("\nEssential structural checks passed.")

,Check,Count
0,Missing user IDs,0
1,Duplicated user IDs,0
2,Negative follower counts,0
3,Negative following counts,0
4,Future creation timestamps,0
5,Duplicated records,0



Missing value comparison:


,original_missing,cleaned_missing,cleaned_missing_percentage
id_user,0,0,0.00
timestamp_user_creation,465337,465337,95.49
username,18,18,0.00
follower_count,465337,465337,95.49
following_count,465337,465337,95.49
profile_url,0,0,0.00
finished_follower_scrape,465337,465337,95.49
finished_following_scrape,465337,465337,95.49
finished_truth_scrape,465337,465337,95.49



Final dataset shape:
(487292, 13)

Essential structural checks passed.


In [19]:
# Identify missing usernames before recovery
df_users_clean["username_was_missing"] = (
    df_users_clean["username"].isna()
)

# Extract usernames from valid profile URLs
extracted_usernames = (
    df_users_clean["profile_url"]
    .str.extract(
        r"(?i)^https?://(?:www\.)?truthsocial\.com/@([^/?#]+)/?$",
        expand=False
    )
)

# Identify recoverable missing usernames
recoverable = (
    df_users_clean["username"].isna()
    & extracted_usernames.notna()
)

# Recover usernames
df_users_clean.loc[
    recoverable, "username"
] = extracted_usernames.loc[recoverable]

# Track imputed values
df_users_clean["username_recovered_from_url"] = recoverable

# Summarize recovery
print("Originally missing usernames:",
      df_users_clean["username_was_missing"].sum())

print("Recovered usernames:",
      recoverable.sum())

print("Remaining missing usernames:",
      df_users_clean["username"].isna().sum())

print("\nRecovered records:")

display(
    df_users_clean.loc[
        recoverable,
        ["id_user", "username", "profile_url"]
    ]
)

Originally missing usernames: 18
Recovered usernames: 18
Remaining missing usernames: 0

Recovered records:


,id_user,username,profile_url
119490,119560,nan,https://truthsocial.com/@nan
127811,127881,nan,https://truthsocial.com/@nan
178390,178460,nan,https://truthsocial.com/@nan
210208,210278,nan,https://truthsocial.com/@nan
275812,275882,nan,https://truthsocial.com/@nan
277411,277481,nan,https://truthsocial.com/@nan
280255,280325,nan,https://truthsocial.com/@nan
284401,284471,nan,https://truthsocial.com/@nan
319040,319110,nan,https://truthsocial.com/@nan
335832,335902,nan,https://truthsocial.com/@nan


In [20]:
# Correct invalid username recovery

# Identify invalid recovered usernames
invalid_recovery = (
    df_users_clean["username_recovered_from_url"]
    & df_users_clean["username"].str.lower().isin(
        ["nan", "none", "null", "<na>", "-1"]
    )
)

# Restore missing values
df_users_clean.loc[
    invalid_recovery, "username"
] = pd.NA

# Restore missing profile URLs when they contain placeholders
invalid_profile_urls = (
    df_users_clean["profile_url"]
    .str.lower()
    .str.contains(
        r"/@(nan|none|null|<na>|-1)(?:[/?#]|$)",
        regex=True,
        na=False
    )
)

df_users_clean.loc[
    invalid_profile_urls, "profile_url"
] = pd.NA

# Correct recovery tracking
df_users_clean.loc[
    invalid_recovery,
    "username_recovered_from_url"
] = False

print("Invalid recoveries reverted:", invalid_recovery.sum())

print("Remaining missing usernames:",
      df_users_clean["username"].isna().sum())

print("Remaining invalid profile URLs:",
      invalid_profile_urls.sum())

C:\Users\bucco\AppData\Local\Temp\ipykernel_30916\2308765478.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


Invalid recoveries reverted: 18
Remaining missing usernames: 18
Remaining invalid profile URLs: 18


In [21]:
# Remove network-derived and temporary cleaning attributes

original_columns = df_users_raw.columns.tolist()

# Restore the original schema after cleaning
df_users_clean = df_users_clean[
    original_columns
].copy()

# Normalize invalid placeholder profile URLs
invalid_profile_urls = (
    df_users_clean["profile_url"]
    .astype("string")
    .str.fullmatch(
        r"(?i)https?://(?:www\.)?truthsocial\.com/@(?:nan|none|null|<na>|-1)/?"
    )
    .fillna(False)
)

df_users_clean.loc[
    invalid_profile_urls, "profile_url"
] = pd.NA

# Restore invalid username imputations
invalid_usernames = (
    df_users_clean["username"]
    .astype("string")
    .str.lower()
    .isin(["nan", "none", "null", "<na>", "-1"])
)

df_users_clean.loc[
    invalid_usernames, "username"
] = pd.NA

# Validate final structure
assert df_users_clean.columns.tolist() == original_columns
assert len(df_users_clean) == len(df_users_raw)

print("FINAL USERS DATASET")
print("-" * 40)

print("Shape:", df_users_clean.shape)

print("\nMissing values:")
display(df_users_clean.isna().sum().to_frame("Missing"))

print("\nColumns:")
print(df_users_clean.columns.tolist())

FINAL USERS DATASET
----------------------------------------
Shape: (487292, 9)

Missing values:


,Missing
id_user,0
timestamp_user_creation,465337
username,18
follower_count,465337
following_count,465337
profile_url,18
finished_follower_scrape,465337
finished_following_scrape,465337
finished_truth_scrape,465337



Columns:
['id_user', 'timestamp_user_creation', 'username', 'follower_count', 'following_count', 'profile_url', 'finished_follower_scrape', 'finished_following_scrape', 'finished_truth_scrape']


In [22]:
from pathlib import Path
import pandas as pd

# Load the cleaned Truths dataset
TRUTHS_CLEAN_PATH = (
    Path(r"C:\Users\bucco\Desktop\Laboratory of Data Science")
    / "project"
    / "code_project"
    / "data"
    / "Project_Data"
    / "truths_clean.csv"
)

df_truths_ref = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    dtype={"url": "string"},
    low_memory=False
)

# Normalize author identifiers
df_truths_ref["author"] = pd.to_numeric(
    df_truths_ref["author"],
    errors="coerce"
).astype("Int64")

# Extract usernames from canonical post URLs
df_truths_ref["username_from_url"] = (
    df_truths_ref["url"]
    .str.extract(
        r"(?i)^https?://(?:www\.)?truthsocial\.com/@([^/?#]+)/posts/",
        expand=False
    )
    .str.strip()
    .str.lower()
)

# Reject invalid placeholders
invalid_values = [
    "nan", "none", "null", "<na>", "-1", ""
]

df_truths_ref["username_from_url"] = (
    df_truths_ref["username_from_url"]
    .mask(
        df_truths_ref["username_from_url"].isin(
            invalid_values
        )
    )
)

# Identify users with missing usernames
missing_user_ids = (
    df_users_clean.loc[
        df_users_clean["username"].isna(),
        "id_user"
    ]
)

# Find their posts
matching_posts = df_truths_ref[
    df_truths_ref["author"].isin(missing_user_ids)
].copy()

# Count distinct username candidates per user
candidate_summary = (
    matching_posts.groupby("author")
    .agg(
        total_posts=("author", "size"),
        posts_with_username=(
            "username_from_url", "count"
        ),
        distinct_usernames=(
            "username_from_url", "nunique"
        )
    )
)

# Include users without matching posts
candidate_summary = candidate_summary.reindex(
    missing_user_ids.tolist(),
    fill_value=0
)

# Display recovery potential
print("USERNAME RECOVERY ANALYSIS")
print("-" * 40)

print("Users with missing usernames:",
      len(missing_user_ids))

print("Users with matching posts:",
      (candidate_summary["total_posts"] > 0).sum())

print("Users with exactly one candidate username:",
      (candidate_summary["distinct_usernames"] == 1).sum())

print("Users with ambiguous usernames:",
      (candidate_summary["distinct_usernames"] > 1).sum())

print("Users without usable username candidates:",
      (candidate_summary["distinct_usernames"] == 0).sum())

print("\nCandidate summary:")
display(candidate_summary)

print("\nCandidate usernames:")
display(
    matching_posts[
        ["author", "username_from_url", "url"]
    ]
    .dropna(subset=["username_from_url"])
    .drop_duplicates(
        subset=["author", "username_from_url"]
    )
    .sort_values("author")
)

USERNAME RECOVERY ANALYSIS
----------------------------------------
Users with missing usernames: 18
Users with matching posts: 0
Users with exactly one candidate username: 0
Users with ambiguous usernames: 0
Users without usable username candidates: 18

Candidate summary:


,total_posts,posts_with_username,distinct_usernames
author,,,
119560,0,0,0
127881,0,0,0
178460,0,0,0
210278,0,0,0
275882,0,0,0
277481,0,0,0
280325,0,0,0
284471,0,0,0
319110,0,0,0



Candidate usernames:


,author,username_from_url,url


In [ ]:
from pathlib import Path
import pandas as pd

# Define output directory
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

# Preserve only the original columns
original_columns = df_users_raw.columns.tolist()

df_users_final = df_users_clean[
    original_columns
].copy()

# Normalize invalid username placeholders
invalid_usernames = (
    df_users_final["username"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(["nan", "none", "null", "<na>", "-1", ""])
)

df_users_final.loc[
    invalid_usernames, "username"
] = pd.NA

# Normalize invalid profile URL placeholders
invalid_urls = (
    df_users_final["profile_url"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"(?i)https?://(?:www\.)?truthsocial\.com/@(?:nan|none|null|<na>|-1)/?"
    )
    .fillna(False)
)

df_users_final.loc[
    invalid_urls, "profile_url"
] = pd.NA

# Validate final dataset
assert df_users_final.shape == (487292, 9)
assert df_users_final["id_user"].notna().all()
assert df_users_final["id_user"].is_unique

# Define export path
USERS_CLEAN_PATH = DATA_DIR / "users_clean.csv"

# Export cleaned dataset
df_users_final.to_csv(
    USERS_CLEAN_PATH,
    index=False
)

print("EXPORT COMPLETED")
print("-" * 40)

print("File:", USERS_CLEAN_PATH)

print("\nFinal dataset shape:")
print(df_users_final.shape)

print("\nRemaining missing values:")
display(
    df_users_final.isna().sum().to_frame("Missing")
)

print("\nFinal dataset preview:")
display(df_users_final.head())

EXPORT COMPLETED
----------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\users_clean.csv

Final dataset shape:
(487292, 9)

Remaining missing values:


,Missing
id_user,0
timestamp_user_creation,465337
username,18
follower_count,465337
following_count,465337
profile_url,18
finished_follower_scrape,465337
finished_following_scrape,465337
finished_truth_scrape,465337



Final dataset preview:


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape
0,1238,2022-02-01,jeffgolfs,886,204,https://truthsocial.com/@jeffgolfs,False,False,False
1,50,2022-02-01,MizW,2365,2929,https://truthsocial.com/@MizW,False,False,False
2,6951,2022-02-01,SusanCreech,870,1232,https://truthsocial.com/@SusanCreech,False,False,False
3,9861,2022-02-01,BMF6FeeTDeeP,1086,1654,https://truthsocial.com/@BMF6FeeTDeeP,False,False,False
4,34,2022-02-01,DC_Draino,286146,81,https://truthsocial.com/@DC_Draino,True,True,True
